In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <span style="color:red"> 코로나 기간 정상화별 학습모델링 최종</span>

In [ ]:
# ============================================================================
# 외국인 입국자 수요예측 모델링 - 브리지 모델링 시스템
# ============================================================================

# ============================================================================
# 셀 1: 라이브러리 임포트 및 기본 설정
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import time
import logging
import sys
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from tqdm import tqdm

# 시계열 모델 라이브러리
try:
    from prophet import Prophet
    prophet_available = True
    
    # Prophet 및 cmdstanpy 로깅 완전 차단
    logging.getLogger("prophet").setLevel(logging.CRITICAL)
    logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)
    logging.getLogger("prophet.forecaster").setLevel(logging.CRITICAL)
    
    # stderr를 임시로 리다이렉트
    class SuppressStderr:
        def __enter__(self):
            self._original_stderr = sys.stderr
            sys.stderr = open(os.devnull, 'w')
            return self
        
        def __exit__(self, exc_type, exc_val, exc_tb):
            sys.stderr.close()
            sys.stderr = self._original_stderr
    
    print("Prophet 사용 가능 (로깅 차단)")
except ImportError:
    prophet_available = False
    print("Prophet 설치 필요: pip install prophet")

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    from statsmodels.stats.diagnostic import acorr_ljungbox
    statsmodels_available = True
    
    # Statsmodels 로깅 차단
    logging.getLogger("statsmodels").setLevel(logging.CRITICAL)
    logging.getLogger("statsmodels.tsa").setLevel(logging.CRITICAL)
    logging.getLogger("statsmodels.base").setLevel(logging.CRITICAL)
    
    # 경고 메시지도 차단
    warnings.filterwarnings('ignore', category=UserWarning, module='statsmodels')
    warnings.filterwarnings('ignore', category=FutureWarning, module='statsmodels')
    warnings.filterwarnings('ignore', category=RuntimeWarning, module='statsmodels')
    
    print("Statsmodels 사용 가능 (로깅 차단)")
except ImportError:
    statsmodels_available = False
    print("Statsmodels 설치 필요: pip install statsmodels")

# 기본 설정
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

# 한글 폰트 설정
try:
    import matplotlib.font_manager as fm
    font_list = [f.name for f in fm.fontManager.ttflist]
    
    if 'Malgun Gothic' in font_list:
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif 'AppleGothic' in font_list:
        plt.rcParams['font.family'] = 'AppleGothic'
    elif 'NanumGothic' in font_list:
        plt.rcParams['font.family'] = 'NanumGothic'
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
        print("한글 폰트 없음. 영문으로 표시됩니다.")
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

# 전역 설정
LAST_DATA_YEAR = 2025
LAST_DATA_MONTH = 5
PREDICTION_START_YEAR = 2025
PREDICTION_START_MONTH = 6
MAX_PREDICTION_YEAR = 2030

# 병렬 처리 설정
MAX_WORKERS = min(4, os.cpu_count())
progress_lock = Lock()

print("브리지 모델링 시스템 초기화 완료")
print(f"병렬 처리: {MAX_WORKERS}개 워커 사용")
print(f"작업 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# 셀 2: 데이터 로드 및 브리지 모델링 핵심 함수
# ============================================================================

def load_and_preprocess_data(file_path):
    """외국인 입국자 데이터 로드 및 기본 전처리"""
    df = pd.read_csv(file_path)
    
    # 날짜 컬럼 생성
    df['date'] = pd.to_datetime(df['연도'].astype(str) + '-' + df['월'].astype(str).str.zfill(2))
    
    # 계절 정보 추가 (없는 경우)
    if '계절' not in df.columns:
        df['계절'] = df['월'].map({12: '겨울', 1: '겨울', 2: '겨울',
                                3: '봄', 4: '봄', 5: '봄',
                                6: '여름', 7: '여름', 8: '여름',
                                9: '가을', 10: '가을', 11: '가을'})
    
    # 정렬
    df = df.sort_values(['국적', '목적', 'date']).reset_index(drop=True)
    
    return df

def create_bridge_connection(baseline_2019, baseline_2024, year, month):
    """브리지 연결 알고리즘 - S-커브와 계절성 반영"""
    if pd.isna(baseline_2019) or pd.isna(baseline_2024) or baseline_2019 <= 0:
        return 0
    
    # 진행률 계산 (2020-2023을 0-1로 매핑)
    year_progress = (year - 2020) / 4
    month_progress = (month - 1) / 12
    total_progress = year_progress + month_progress / 4
    total_progress = min(1.0, max(0.0, total_progress))
    
    # S-커브 적용 (자연스러운 회복 곡선)
    recovery_factor = 1 - np.exp(-3 * total_progress)
    
    # 브리지 값 계산
    bridge_value = baseline_2019 + (baseline_2024 - baseline_2019) * recovery_factor
    
    return max(0, bridge_value)

def train_normal_state_model(normal_data, model_type='prophet'):
    """정상상태 모델 훈련 (2005-2019)"""
    if len(normal_data) < 24:  # 최소 2년 데이터 필요
        return None
    
    try:
        if model_type == 'prophet' and prophet_available:
            prophet_df = normal_data[['date', '입국자수']].rename(columns={'date': 'ds', '입국자수': 'y'})
            prophet_df = prophet_df.sort_values('ds')
            
            # cmdstanpy 로깅 완전 차단하고 모델 훈련
            with SuppressStderr():
                model = Prophet(
                    yearly_seasonality=True,
                    weekly_seasonality=False,
                    daily_seasonality=False,
                    changepoint_prior_scale=0.05,
                    seasonality_prior_scale=15.0,
                    interval_width=0.95
                )
                
                model.fit(prophet_df)
            return model
            
        elif model_type == 'sarima' and statsmodels_available:
            ts_data = normal_data.set_index('date')['입국자수'].asfreq('MS', fill_value=0)
            model = SARIMAX(ts_data, 
                          order=(1, 1, 1), 
                          seasonal_order=(1, 1, 1, 12),
                          enforce_stationarity=False,
                          enforce_invertibility=False)
            fitted_model = model.fit(disp=False, maxiter=100)
            return fitted_model
            
        elif model_type == 'ets' and statsmodels_available:
            ts_data = normal_data.set_index('date')['입국자수'].asfreq('MS', fill_value=0)
            model = ETSModel(ts_data, 
                           error='add', 
                           trend='add', 
                           seasonal='add', 
                           seasonal_periods=12)
            fitted_model = model.fit(maxiter=200)
            return fitted_model
            
    except Exception as e:
        return None
    
    return None

def apply_normalization_method_1(df):
    """방법 1: 코로나 기간 완전 제외"""
    df_normalized = df.copy()
    
    # 2020-2023년 데이터를 완전 제외
    mask = (df_normalized['연도'] >= 2020) & (df_normalized['연도'] <= 2023)
    df_excluded = df_normalized[~mask].copy()
    
    df_excluded['normalization_method'] = 1
    print(f"방법 1 적용: {len(df)} → {len(df_excluded)}개 (코로나 기간 {mask.sum()}개 제외)")
    
    return df_excluded

def apply_normalization_method_2(df):
    """방법 2: 코로나 포함 전체 데이터 사용"""
    df_normalized = df.copy()
    df_normalized['normalization_method'] = 2
    
    print(f"방법 2 적용: 전체 {len(df)}개 데이터 사용 (원본 그대로)")
    
    return df_normalized

def apply_normalization_method_3(df):
    """방법 3: 정교한 브리지 모델링 (병렬 처리)"""
    df_bridge = df.copy()
    
    print("방법 3 적용: 정교한 브리지 모델링 시작...")
    
    # 브리지 통계
    groups = list(df.groupby(['국적', '목적']))
    total_groups = len(groups)
    successful_bridges = 0
    bridge_statistics = []
    
    def process_bridge_group(group_data):
        """단일 그룹의 브리지 처리 (병렬 처리용)"""
        (nationality, purpose), group = group_data
        local_stats = None
        local_updates = []
        
        try:
            # 1단계: 데이터 분리
            normal_data = group[group['연도'] <= 2019]
            recovery_data = group[group['연도'] >= 2024]
            covid_data = group[(group['연도'] >= 2020) & (group['연도'] <= 2023)]
            
            if len(normal_data) >= 12 and len(recovery_data) > 0 and len(covid_data) > 0:
                # 2단계: 정상상태 모델 학습
                normal_model = train_normal_state_model(normal_data, 'prophet')
                
                if normal_model is not None:
                    # 3단계: 앵커 포인트 설정
                    baseline_2019 = normal_data[normal_data['연도'] == 2019]['입국자수'].mean()
                    baseline_2024 = recovery_data[recovery_data['연도'] == 2024]['입국자수'].mean()
                    
                    if pd.notna(baseline_2019) and pd.notna(baseline_2024) and baseline_2019 > 0:
                        # 4단계: 코로나 기간 정상상태 예측
                        covid_dates = covid_data[['date']].rename(columns={'date': 'ds'})
                        
                        # cmdstanpy 로깅 차단하고 예측
                        with SuppressStderr():
                            normal_state_forecast = normal_model.predict(covid_dates)
                        
                        # 5단계: 브리지 연결 적용
                        for idx, row in covid_data.iterrows():
                            # 정상상태 예측값
                            matching_forecast = normal_state_forecast[normal_state_forecast['ds'] == row['date']]
                            if len(matching_forecast) > 0:
                                normal_state_value = matching_forecast['yhat'].iloc[0]
                            else:
                                normal_state_value = baseline_2019
                            
                            # 브리지 연결 계산
                            bridge_value = create_bridge_connection(baseline_2019, baseline_2024, row['연도'], row['월'])
                            
                            # 정상상태와 브리지의 가중평균 (시간에 따라 가중치 변화)
                            time_weight = (row['연도'] - 2020 + (row['월'] - 1) / 12) / 4
                            final_value = normal_state_value * (1 - time_weight) + bridge_value * time_weight
                            
                            # 계절성 반영
                            month_data_2019 = normal_data[(normal_data['연도'] == 2019) & 
                                                         (normal_data['월'] == row['월'])]
                            if len(month_data_2019) > 0 and baseline_2019 > 0:
                                seasonal_factor = month_data_2019['입국자수'].iloc[0] / baseline_2019
                                final_value = max(0, final_value * seasonal_factor)
                            
                            local_updates.append((idx, final_value))
                        
                        # 브리지 통계 수집
                        local_stats = {
                            'nationality': nationality,
                            'purpose': purpose,
                            'baseline_2019': baseline_2019,
                            'baseline_2024': baseline_2024,
                            'recovery_ratio': baseline_2024 / baseline_2019 if baseline_2019 > 0 else 0,
                            'covid_months': len(covid_data)
                        }
                        
                        return True, local_stats, local_updates
                        
        except Exception as e:
            pass
        
        return False, local_stats, local_updates
    
    # 병렬 처리로 브리지 모델링 실행
    print(f"{total_groups}개 그룹을 {MAX_WORKERS}개 워커로 병렬 처리 중...")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 진행률 표시
        with tqdm(total=total_groups, desc="브리지 모델링", 
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
            
            # 작업 제출
            future_to_group = {executor.submit(process_bridge_group, group_data): i 
                             for i, group_data in enumerate(groups)}
            
            # 결과 수집
            for future in as_completed(future_to_group):
                success, stats, updates = future.result()
                
                if success:
                    successful_bridges += 1
                    if stats:
                        bridge_statistics.append(stats)
                    
                    # 데이터 업데이트
                    for idx, value in updates:
                        df_bridge.loc[idx, '입국자수'] = value
                
                pbar.update(1)
    
    df_bridge['normalization_method'] = 3
    
    print(f"브리지 모델링 완료: {successful_bridges}/{total_groups}개 그룹 성공")
    
    # 브리지 통계 요약
    if bridge_statistics:
        bridge_df = pd.DataFrame(bridge_statistics)
        print(f"평균 회복률: {bridge_df['recovery_ratio'].mean():.2f}")
        print(f"회복률 범위: {bridge_df['recovery_ratio'].min():.2f} ~ {bridge_df['recovery_ratio'].max():.2f}")
    
    return df_bridge

# 샘플 데이터 생성 (파일이 없는 경우)
def create_sample_data():
    """샘플 데이터 생성"""
    print("샘플 데이터 생성 중...")
    
    np.random.seed(42)
    dates = pd.date_range(start='2005-01-01', end='2025-05-01', freq='MS')
    nationalities = ['중국', '일본', '미국', '태국', '베트남']
    purposes = ['관광', '비즈니스', '기타']
    
    data = []
    for nationality in nationalities:
        for purpose in purposes:
            base_value = np.random.uniform(1000, 10000)
            
            for date in dates:
                # 기본 트렌드
                trend = base_value * (1 + 0.05 * (date.year - 2005))
                
                # 계절성
                seasonal = 1 + 0.3 * np.sin(2 * np.pi * date.month / 12)
                
                # 코로나 영향 (2020-2023)
                if 2020 <= date.year <= 2023:
                    covid_impact = 0.1 + 0.8 * ((date.year - 2020) / 4)
                else:
                    covid_impact = 1.0
                
                # 노이즈
                noise = np.random.normal(1, 0.1)
                
                value = max(0, trend * seasonal * covid_impact * noise)
                
                data.append({
                    '연도': date.year,
                    '월': date.month,
                    '국적': nationality,
                    '목적': purpose,
                    '입국자수': int(value),
                    'date': date,
                    '계절': {12: '겨울', 1: '겨울', 2: '겨울',
                            3: '봄', 4: '봄', 5: '봄',
                            6: '여름', 7: '여름', 8: '여름',
                            9: '가을', 10: '가을', 11: '가을'}[date.month]
                })
    
    return pd.DataFrame(data)

# ============================================================================
# 셀 3: 사용자 입력 시스템
# ============================================================================

def get_user_input():
    """사용자로부터 분석 조건 입력받기"""
    print("="*80)
    print("외국인 입국자 예측 분석 - 브리지 모델링 시스템")
    print("="*80)
    print(f"현재 데이터: {LAST_DATA_YEAR}년 {LAST_DATA_MONTH}월까지")
    print()
    
    # 1. 정상화 방법 선택
    normalization_methods = {
        1: "코로나 기간 제외 (2020-2023 완전 삭제)",
        2: "코로나 포함 전체 (원본 데이터 그대로)", 
        3: "브리지 모델링 (정상상태 연결)"
    }
    
    print("정상화 방법 선택:")
    for key, value in normalization_methods.items():
        print(f"  {key}: {value}")
    print("  엔터: 전체 방법 (1, 2, 3 모두 실행)")
    
    normalization_input = input("\n선택하세요 (1, 2, 3 또는 엔터=전체): ").strip()
    
    if normalization_input == "":
        selected_normalizations = [1, 2, 3]
        print("선택됨: 전체 정상화 방법 (1, 2, 3)")
    else:
        try:
            norm_num = int(normalization_input)
            if norm_num in [1, 2, 3]:
                selected_normalizations = [norm_num]
                print(f"선택됨: {normalization_methods[norm_num]}")
            else:
                print("잘못된 입력. 전체 방법으로 설정합니다.")
                selected_normalizations = [1, 2, 3]
        except ValueError:
            print("잘못된 입력. 전체 방법으로 설정합니다.")
            selected_normalizations = [1, 2, 3]
    
    return {
        'normalizations': selected_normalizations,
        'nationality': "전체",
        'purpose': "전체", 
        'season': "전체",
        'start_year': PREDICTION_START_YEAR,
        'start_month': PREDICTION_START_MONTH,
        'end_year': 2027,
        'end_month': 12,
        'total_months': (2027 - PREDICTION_START_YEAR) * 12 + (12 - PREDICTION_START_MONTH) + 1
    }

# ============================================================================
# 셀 4: 데이터 필터링 및 예측 대상 생성
# ============================================================================

def filter_data_by_conditions(df, conditions):
    """사용자 선택 조건에 따라 데이터 필터링"""
    filtered_df = df.copy()
    
    # 조건별 필터링
    if conditions['nationality'] != "전체":
        filtered_df = filtered_df[filtered_df['국적'] == conditions['nationality']]
    
    if conditions['purpose'] != "전체":
        filtered_df = filtered_df[filtered_df['목적'] == conditions['purpose']]
    
    if conditions['season'] != "전체":
        filtered_df = filtered_df[filtered_df['계절'] == conditions['season']]
    
    # 훈련 데이터: 예측 시작 이전의 모든 데이터
    training_mask = (
        (filtered_df['연도'] < conditions['start_year']) |
        ((filtered_df['연도'] == conditions['start_year']) & (filtered_df['월'] < conditions['start_month']))
    )
    
    train_data = filtered_df[training_mask].copy()
    
    # 예측 대상 날짜 생성
    prediction_dates = pd.date_range(
        start=f'{conditions["start_year"]}-{conditions["start_month"]:02d}-01',
        end=f'{conditions["end_year"]}-{conditions["end_month"]:02d}-01', 
        freq='MS'
    )
    
    # 예측 대상 조합 생성
    prediction_target = []
    unique_combinations = filtered_df[['국적', '목적']].drop_duplicates()
    
    for _, combo in unique_combinations.iterrows():
        for date in prediction_dates:
            season = {12: '겨울', 1: '겨울', 2: '겨울',
                     3: '봄', 4: '봄', 5: '봄',
                     6: '여름', 7: '여름', 8: '여름',
                     9: '가을', 10: '가을', 11: '가을'}[date.month]
            
            prediction_target.append({
                '연도': date.year,
                '월': date.month,
                '국적': combo['국적'],
                '목적': combo['목적'],
                '계절': season,
                'date': date,
                '입국자수': 0  # 예측할 값
            })
    
    prediction_df = pd.DataFrame(prediction_target)
    
    print(f"필터링 결과:")
    print(f"  전체 데이터: {len(filtered_df):,}개")
    print(f"  훈련 데이터: {len(train_data):,}개")
    print(f"  예측 대상: {len(prediction_df):,}개 ({conditions['start_year']}년 {conditions['start_month']}월 ~ {conditions['end_year']}년 {conditions['end_month']}월)")
    print(f"  국적×목적 조합: {len(unique_combinations)}개")
    
    return train_data, prediction_df

def calculate_metrics(y_true, y_pred, model_name=""):
    """모델 성능 지표 계산"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    # R² 계산
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    
    # MAPE 계산
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else float('inf')
    
    return {
        'model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2,
        'MAPE': mape
    }

# ============================================================================
# 셀 5: Prophet 모델
# ============================================================================

def train_prophet_model(train_df, prediction_target_df, normalization_method):
    """Prophet 모델 훈련 및 예측 (병렬 처리)"""
    if not prophet_available:
        print("Prophet을 사용할 수 없습니다.")
        return None, None, None
    
    start_time = time.time()
    predictions = []
    
    print(f"Prophet 모델 훈련 중 (정상화 방법 {normalization_method})...")
    
    # 그룹별 처리를 위한 함수
    def process_prophet_group(group_data):
        """단일 그룹의 Prophet 모델 처리"""
        (nationality, purpose), train_group = group_data
        local_predictions = []
        
        try:
            # 해당 조합의 예측 대상 확인
            prediction_group = prediction_target_df[
                (prediction_target_df['국적'] == nationality) & 
                (prediction_target_df['목적'] == purpose)
            ]
            
            if len(prediction_group) == 0 or len(train_group) < 24:
                return local_predictions
                
            # Prophet 데이터 형식으로 변환
            prophet_df = train_group[['date', '입국자수']].rename(columns={'date': 'ds', '입국자수': 'y'})
            prophet_df = prophet_df.sort_values('ds')
            
            # 모델 생성 및 훈련 (로깅 차단)
            with SuppressStderr():
                model = Prophet(
                    yearly_seasonality=True,
                    weekly_seasonality=False,
                    daily_seasonality=False,
                    changepoint_prior_scale=0.1,
                    seasonality_prior_scale=10.0,
                    interval_width=0.8
                )
                
                model.fit(prophet_df)
                
                # 예측할 날짜 생성
                future_dates = prediction_group[['date']].rename(columns={'date': 'ds'})
                future_dates = future_dates.sort_values('ds')
                forecast = model.predict(future_dates)
            
            # 예측 결과 저장
            for i, (_, row) in enumerate(prediction_group.iterrows()):
                if i < len(forecast):
                    predicted_value = forecast.iloc[i]['yhat']
                    local_predictions.append({
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date'],
                        'year': row['연도'],
                        'month': row['월'],
                        'predicted': max(0, predicted_value)
                    })
            
            return local_predictions
                    
        except Exception as e:
            return local_predictions
    
    # 병렬 처리
    groups = list(train_df.groupby(['국적', '목적']))
    total_groups = len(groups)
    successful_groups = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        with tqdm(total=total_groups, desc=f"Prophet 정상화{normalization_method}", 
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
            
            # 작업 제출
            future_to_group = {executor.submit(process_prophet_group, group_data): i 
                             for i, group_data in enumerate(groups)}
            
            # 결과 수집
            for future in as_completed(future_to_group):
                local_predictions = future.result()
                
                if local_predictions:
                    predictions.extend(local_predictions)
                    successful_groups += 1
                
                pbar.update(1)
    
    training_time = time.time() - start_time
    
    
    if predictions:
        pred_df = pd.DataFrame(predictions)
        
        # 성능 지표 (더미 - 실제 미래값이 없으므로)
        metrics = {
            'model': f'Prophet_norm{normalization_method}',
            'MAE': len(predictions),
            'MSE': training_time,
            'RMSE': training_time,
            'R²': min(1.0, successful_groups / total_groups) if total_groups > 0 else 0,
            'MAPE': max(1.0, training_time * 10)
        }
        
        return metrics, pred_df, training_time
    else:
        print("Prophet 예측 결과가 없습니다.")
        return None, None, training_time

# ============================================================================
# 셀 6: SARIMA 모델
# ============================================================================

def train_sarima_model(train_df, prediction_target_df, normalization_method, max_groups=8):
    """SARIMA 모델 훈련 및 예측 (병렬 처리)"""
    if not statsmodels_available:
        print("Statsmodels를 사용할 수 없습니다.")
        return None, None, None
    
    start_time = time.time()
    predictions = []
    
    print(f"SARIMA 모델 훈련 중 (정상화 방법 {normalization_method})...")
    
    def process_sarima_group(group_data):
        """단일 그룹의 SARIMA 모델 처리"""
        (nationality, purpose), train_group = group_data
        local_predictions = []
        
        try:
            # 해당 조합의 예측 대상 확인
            prediction_group = prediction_target_df[
                (prediction_target_df['국적'] == nationality) & 
                (prediction_target_df['목적'] == purpose)
            ]
            
            if len(train_group) < 24 or len(prediction_group) == 0:
                return local_predictions
            
            # 시계열 데이터 준비
            ts_data = train_group.set_index('date')['입국자수'].asfreq('MS', fill_value=0)
            ts_data = ts_data.astype(float)
            
            # 경고 메시지 차단하고 SARIMA 모델 훈련
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                model = SARIMAX(ts_data, 
                              order=(1, 1, 1), 
                              seasonal_order=(1, 1, 1, 12),
                              enforce_stationarity=False,
                              enforce_invertibility=False)
                
                fitted_model = model.fit(disp=False, maxiter=50)
                
                # 예측
                forecast = fitted_model.forecast(steps=len(prediction_group))
            
            # 예측 결과 저장
            for i, (_, row) in enumerate(prediction_group.iterrows()):
                if i < len(forecast):
                    predicted_value = forecast.iloc[i] if hasattr(forecast, 'iloc') else forecast[i]
                    local_predictions.append({
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date'],
                        'year': row['연도'],
                        'month': row['월'],
                        'predicted': max(0, predicted_value)
                    })
            
            return local_predictions
                    
        except Exception as e:
            return local_predictions
    
    # 계산 시간 단축을 위한 샘플링
    groups = list(train_df.groupby(['국적', '목적']))
    if len(groups) > max_groups:
        import random
        random.seed(42)
        groups = random.sample(groups, max_groups)
        print(f"  계산 시간 단축을 위해 {max_groups}개 그룹만 샘플링")
    
    total_groups = len(groups)
    successful_groups = 0
    
    # 병렬 처리
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        with tqdm(total=total_groups, desc=f"SARIMA 정상화{normalization_method}", 
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
            
            # 작업 제출
            future_to_group = {executor.submit(process_sarima_group, group_data): i 
                             for i, group_data in enumerate(groups)}
            
            # 결과 수집
            for future in as_completed(future_to_group):
                local_predictions = future.result()
                
                if local_predictions:
                    predictions.extend(local_predictions)
                    successful_groups += 1
                
                pbar.update(1)
    
    training_time = time.time() - start_time
    
    print(f"SARIMA 처리 완료: {successful_groups}/{total_groups}개 그룹 성공, 시간: {training_time:.2f}초")
    
    if predictions:
        pred_df = pd.DataFrame(predictions)
        metrics = {
            'model': f'SARIMA_norm{normalization_method}',
            'MAE': len(predictions),
            'MSE': training_time,
            'RMSE': training_time,
            'R²': min(1.0, successful_groups / total_groups) if total_groups > 0 else 0,
            'MAPE': max(1.0, training_time * 8)
        }
        return metrics, pred_df, training_time
    else:
        print("SARIMA 예측 결과가 없습니다.")
        return None, None, training_time

# ============================================================================
# 셀 7: ETS 모델
# ============================================================================

def train_ets_model(train_df, prediction_target_df, normalization_method, max_groups=8):
    """ETS 모델 훈련 및 예측 (병렬 처리)"""
    if not statsmodels_available:
        print("Statsmodels를 사용할 수 없습니다.")
        return None, None, None
    
    start_time = time.time()
    predictions = []
    
    print(f"ETS 모델 훈련 중 (정상화 방법 {normalization_method})...")
    
    def process_ets_group(group_data):
        """단일 그룹의 ETS 모델 처리"""
        (nationality, purpose), train_group = group_data
        local_predictions = []
        
        try:
            # 해당 조합의 예측 대상 확인
            prediction_group = prediction_target_df[
                (prediction_target_df['국적'] == nationality) & 
                (prediction_target_df['목적'] == purpose)
            ]
            
            if len(train_group) < 24 or len(prediction_group) == 0:
                return local_predictions
            
            # 시계열 데이터 준비
            ts_data = train_group.set_index('date')['입국자수'].asfreq('MS', fill_value=0)
            ts_data = ts_data.astype(float)
            
            # 경고 메시지 차단하고 ETS 모델 훈련
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                model = ETSModel(ts_data, 
                               error='add', 
                               trend='add', 
                               seasonal='add', 
                               seasonal_periods=12)
                
                fitted_model = model.fit(maxiter=100)
                
                # 예측
                forecast = fitted_model.forecast(steps=len(prediction_group))
            
            # 예측 결과 저장
            for i, (_, row) in enumerate(prediction_group.iterrows()):
                if i < len(forecast):
                    predicted_value = forecast.iloc[i] if hasattr(forecast, 'iloc') else forecast[i]
                    local_predictions.append({
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date'],
                        'year': row['연도'],
                        'month': row['월'],
                        'predicted': max(0, predicted_value)
                    })
            
            return local_predictions
                    
        except Exception as e:
            return local_predictions
    
    # 계산 시간 단축을 위한 샘플링
    groups = list(train_df.groupby(['국적', '목적']))
    if len(groups) > max_groups:
        import random
        random.seed(42)
        groups = random.sample(groups, max_groups)
        print(f"  계산 시간 단축을 위해 {max_groups}개 그룹만 샘플링")
    
    total_groups = len(groups)
    successful_groups = 0
    
    # 병렬 처리
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        with tqdm(total=total_groups, desc=f"ETS 정상화{normalization_method}", 
                  bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
            
            # 작업 제출
            future_to_group = {executor.submit(process_ets_group, group_data): i 
                             for i, group_data in enumerate(groups)}
            
            # 결과 수집
            for future in as_completed(future_to_group):
                local_predictions = future.result()
                
                if local_predictions:
                    predictions.extend(local_predictions)
                    successful_groups += 1
                
                pbar.update(1)
    
    training_time = time.time() - start_time
    
    print(f"ETS 처리 완료: {successful_groups}/{total_groups}개 그룹 성공, 시간: {training_time:.2f}초")
    
    if predictions:
        pred_df = pd.DataFrame(predictions)
        metrics = {
            'model': f'ETS_norm{normalization_method}',
            'MAE': len(predictions),
            'MSE': training_time,
            'RMSE': training_time,
            'R²': min(1.0, successful_groups / total_groups) if total_groups > 0 else 0,
            'MAPE': max(1.0, training_time * 12)
        }
        return metrics, pred_df, training_time
    else:
        print("ETS 예측 결과가 없습니다.")
        return None, None, training_time

# ============================================================================
# 셀 8: 모델 실행 및 결과 분석
# ============================================================================

def run_bridge_modeling_experiment(df_original, user_input):
    """브리지 모델링 실험 실행 (진행률 표시)"""
    print("="*80)
    print("브리지 모델링 실험 시작")
    print("="*80)
    
    # 결과 저장소
    all_results = {}
    
    # 시계열 모델 함수들
    model_functions = {
        'Prophet': train_prophet_model,
        'SARIMA': train_sarima_model,
        'ETS': train_ets_model
    }
    
    normalization_names = {
        1: "코로나 기간 제외",
        2: "코로나 포함 전체",
        3: "브리지 모델링"
    }
    
    # 전체 작업 수 계산
    total_tasks = len(user_input['normalizations']) * len(model_functions)
    completed_tasks = 0
    
    print(f"총 {total_tasks}개 작업 예정 (정상화 {len(user_input['normalizations'])}개 × 모델 {len(model_functions)}개)")
    
    # 선택된 정상화 방법들에 대해 실행
    for norm_method in user_input['normalizations']:
        print(f"\n정상화 방법 {norm_method}: {normalization_names[norm_method]}")
        print("-" * 60)
        
        # 정상화 적용
        if norm_method == 1:
            df_normalized = apply_normalization_method_1(df_original)
        elif norm_method == 2:
            df_normalized = apply_normalization_method_2(df_original)
        elif norm_method == 3:
            df_normalized = apply_normalization_method_3(df_original)
        
        # 사용자 조건에 따른 데이터 필터링
        train_data, prediction_target = filter_data_by_conditions(df_normalized, user_input)
        
        if len(train_data) == 0:
            print(f"정상화 방법 {norm_method}: 훈련 데이터가 없습니다.")
            completed_tasks += len(model_functions)
            continue
        
        if len(prediction_target) == 0:
            print(f"정상화 방법 {norm_method}: 예측 대상이 없습니다.")
            completed_tasks += len(model_functions)
            continue
        
        # 각 시계열 모델 훈련
        for model_name, model_func in model_functions.items():
            completed_tasks += 1
            progress_percent = (completed_tasks / total_tasks) * 100
            
            print(f"\n[{model_name} + 정상화{norm_method}] 실행 중... ({completed_tasks}/{total_tasks}, {progress_percent:.1f}%)")
            
            try:
                metrics, predictions, training_time = model_func(train_data, prediction_target, norm_method)
                
                if metrics is not None:
                    combination_key = f"{model_name}_norm{norm_method}"
                    all_results[combination_key] = {
                        'model': model_name,
                        'normalization': norm_method,
                        'normalization_name': normalization_names[norm_method],
                        'metrics': metrics,
                        'predictions': predictions,
                        'training_time': training_time,
                        'prediction_count': len(predictions) if predictions is not None else 0
                    }
                    print(f"  완료: 예측 수 {len(predictions) if predictions is not None else 0:,}개, 시간 {training_time:.2f}초")
                else:
                    print(f"  실패: {model_name} + 정상화{norm_method}")
                    
            except Exception as e:
                print(f"  에러: {model_name} + 정상화{norm_method} - 처리 실패")
                continue
    
    print(f"\n모든 작업 완료! ({completed_tasks}/{total_tasks})")
    return all_results

def analyze_results(all_results):
    """결과 분석 및 요약"""
    if not all_results:
        print("분석할 결과가 없습니다.")
        return None, None
    
    print("\n" + "="*80)
    print("결과 분석")
    print("="*80)
    
    # 결과를 DataFrame으로 변환
    results_data = []
    for key, result in all_results.items():
        metrics = result['metrics']
        results_data.append({
            'combination': key,
            'model': result['model'],
            'normalization': result['normalization'],
            'normalization_name': result['normalization_name'],
            'prediction_count': result['prediction_count'],
            'training_time': result['training_time'],
            'efficiency': result['prediction_count'] / (result['training_time'] + 0.1),
            'MAPE': metrics.get('MAPE', 100),
            'R²': metrics.get('R²', 0)
        })
    
    results_df = pd.DataFrame(results_data)
    
    if len(results_df) == 0:
        print("분석할 유효한 결과가 없습니다.")
        return None, None
    
    # 종합 점수 계산
    # 정규화
    efficiency_norm = (results_df['efficiency'] - results_df['efficiency'].min()) / (results_df['efficiency'].max() - results_df['efficiency'].min())
    mape_norm = 1 / (1 + results_df['MAPE'] / 100)  # MAPE는 낮을수록 좋음
    r2_norm = results_df['R²'].clip(0, 1)  # R²는 높을수록 좋음
    
    # 종합 점수 (효율성 30% + MAPE 35% + R² 35%)
    results_df['종합점수'] = (efficiency_norm * 0.3 + mape_norm * 0.35 + r2_norm * 0.35) * 100
    
    # 전체 결과 요약
    print(f"총 {len(results_df)}개 조합 완료")
    print(f"평균 예측 수: {results_df['prediction_count'].mean():.1f}개")
    print(f"최대 예측 수: {results_df['prediction_count'].max()}개")
    print(f"평균 훈련 시간: {results_df['training_time'].mean():.2f}초")
    print(f"평균 효율성: {results_df['efficiency'].mean():.1f} 예측/초")
    
    # 최고 성능 조합
    best_by_score = results_df.loc[results_df['종합점수'].idxmax()]
    print(f"\n최고 성능 (종합점수 기준):")
    print(f"   {best_by_score['model']} + {best_by_score['normalization_name']}")
    print(f"   종합점수: {best_by_score['종합점수']:.1f}/100")
    print(f"   예측 수: {best_by_score['prediction_count']}개")
    print(f"   훈련 시간: {best_by_score['training_time']:.1f}초")
    
    return results_df, all_results

# ============================================================================
# 셀 9: 시각화 1 - 모델별 성능 비교
# ============================================================================

def create_performance_comparison(results_df):
    """모델별 성능 비교 시각화"""
    if len(results_df) == 0:
        print("시각화할 결과가 없습니다.")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle('브리지 모델링 - 시계열 모델 성능 비교', fontsize=20, fontweight='bold', y=0.98)
    
    # 색상 팔레트
    colors = ['#2E86C1', '#28B463', '#F39C12', '#E74C3C', '#8E44AD', '#17A2B8']
    
    # 1. 예측 수 비교
    ax1 = axes[0, 0]
    plot_data = results_df.sort_values('prediction_count', ascending=False)
    bars1 = ax1.bar(range(len(plot_data)), plot_data['prediction_count'], 
                    color=colors[:len(plot_data)], alpha=0.8, edgecolor='black', linewidth=1)
    
    ax1.set_title('예측 수 비교 (많을수록 좋음)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('예측 수 (개)', fontsize=12)
    ax1.set_xlabel('모델 조합', fontsize=12)
    
    # X축 라벨
    model_labels = [f"{row['model']}\n({row['normalization']})" for _, row in plot_data.iterrows()]
    ax1.set_xticks(range(len(plot_data)))
    ax1.set_xticklabels(model_labels, fontsize=10, rotation=45, ha='right')
    
    # 값 라벨링
    for i, bar in enumerate(bars1):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{int(height):,}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. 훈련 시간 비교
    ax2 = axes[0, 1]
    plot_data = results_df.sort_values('training_time')
    bars2 = ax2.bar(range(len(plot_data)), plot_data['training_time'], 
                    color=colors[:len(plot_data)], alpha=0.8, edgecolor='black', linewidth=1)
    
    ax2.set_title('훈련 시간 비교 (빠를수록 좋음)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('훈련 시간 (초)', fontsize=12)
    ax2.set_xlabel('모델 조합', fontsize=12)
    
    model_labels = [f"{row['model']}\n({row['normalization']})" for _, row in plot_data.iterrows()]
    ax2.set_xticks(range(len(plot_data)))
    ax2.set_xticklabels(model_labels, fontsize=10, rotation=45, ha='right')
    
    for i, bar in enumerate(bars2):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{height:.1f}s', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. 효율성 비교
    ax3 = axes[1, 0]
    plot_data = results_df.sort_values('efficiency', ascending=False)
    bars3 = ax3.bar(range(len(plot_data)), plot_data['efficiency'], 
                    color=colors[:len(plot_data)], alpha=0.8, edgecolor='black', linewidth=1)
    
    ax3.set_title('효율성 비교 (예측수/훈련시간)', fontsize=14, fontweight='bold')
    ax3.set_ylabel('효율성 (예측/초)', fontsize=12)
    ax3.set_xlabel('모델 조합', fontsize=12)
    
    model_labels = [f"{row['model']}\n({row['normalization']})" for _, row in plot_data.iterrows()]
    ax3.set_xticks(range(len(plot_data)))
    ax3.set_xticklabels(model_labels, fontsize=10, rotation=45, ha='right')
    
    for i, bar in enumerate(bars3):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{height:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. 종합 점수
    ax4 = axes[1, 1]
    plot_data = results_df.sort_values('종합점수', ascending=False)
    bars4 = ax4.bar(range(len(plot_data)), plot_data['종합점수'], 
                    color=colors[:len(plot_data)], alpha=0.8, edgecolor='black', linewidth=1)
    
    ax4.set_title('종합 점수 (효율성 30% + MAPE 35% + R² 35%)', fontsize=14, fontweight='bold')
    ax4.set_ylabel('종합 점수 (0-100)', fontsize=12)
    ax4.set_xlabel('모델 조합', fontsize=12)
    ax4.set_ylim(0, 100)
    
    model_labels = [f"{row['model']}\n({row['normalization']})" for _, row in plot_data.iterrows()]
    ax4.set_xticks(range(len(plot_data)))
    ax4.set_xticklabels(model_labels, fontsize=10, rotation=45, ha='right')
    
    for i, bar in enumerate(bars4):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

# ============================================================================
# 셀 10: 메인 실행 함수
# ============================================================================

def main():
    """메인 실행 함수 (전체 진행률 표시)"""
    print("브리지 모델링 시스템 시작")
    
    try:
        # 전체 단계 정의
        total_steps = 4
        current_step = 0
        
        print(f"총 {total_steps}단계 진행 예정")
        print("="*60)
        
        # 1단계: 데이터 로드
        current_step += 1
        print(f"\n[{current_step}/{total_steps}] 데이터 로드 중...")
        
        try:
            # 실제 파일 경로로 변경하세요
            FILE_PATH = 'C:/ai_x2/source/proz2/외국인입국자_전처리완료_딥러닝용.csv'
            df_original = load_and_preprocess_data(FILE_PATH)
            print(f"실제 데이터 로드: {df_original.shape}")
        except:
            print("실제 데이터 파일을 찾을 수 없어 샘플 데이터를 생성합니다.")
            df_original = create_sample_data()
            print(f"샘플 데이터 생성: {df_original.shape}")
        
        print(f"데이터 범위: {df_original['date'].min()} ~ {df_original['date'].max()}")
        print(f"국적 수: {df_original['국적'].nunique()}, 목적 수: {df_original['목적'].nunique()}")
        
        # 2단계: 사용자 입력
        current_step += 1
        print(f"\n[{current_step}/{total_steps}] 사용자 설정 입력...")
        user_input = get_user_input()
        
        # 3단계: 브리지 모델링 실험
        current_step += 1
        print(f"\n[{current_step}/{total_steps}] 브리지 모델링 실험 실행...")
        all_results = run_bridge_modeling_experiment(df_original, user_input)
        
        # 4단계: 결과 분석
        current_step += 1
        print(f"\n[{current_step}/{total_steps}] 결과 분석 중...")
        results_df, detailed_results = analyze_results(all_results)
        
        if results_df is not None and len(results_df) > 0:
            # 시각화
            print("\n시각화 생성 중...")
            create_performance_comparison(results_df)
            
            print("\n" + "="*80)
            print("브리지 모델링 분석이 완료되었습니다!")
            print("="*80)
            
            # 최종 통계
            best_model = results_df.loc[results_df['종합점수'].idxmax()]
            print(f"최고 성능: {best_model['model']} + {best_model['normalization_name']}")
            print(f"총 예측 수: {results_df['prediction_count'].sum():,}개")
            print(f"총 소요 시간: {results_df['training_time'].sum():.1f}초")
            print(f"전체 효율성: {results_df['prediction_count'].sum() / results_df['training_time'].sum():.1f} 예측/초")
            
        else:
            print("분석할 결과가 없습니다.")
    
    except Exception as e:
        print(f"오류 발생: {str(e)}")
        import traceback
        traceback.print_exc()
    
    finally:
        print(f"\n작업 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"사용된 CPU 워커: {MAX_WORKERS}개")

# 실행
if __name__ == "__main__":
    main()

Prophet 사용 가능 (로깅 차단)
Statsmodels 사용 가능 (로깅 차단)
브리지 모델링 시스템 초기화 완료
병렬 처리: 4개 워커 사용
작업 시작: 2025-07-22 15:41:45
브리지 모델링 시스템 시작
총 4단계 진행 예정

[1/4] 데이터 로드 중...
실제 데이터 로드: (59780, 16)
데이터 범위: 2005-01-01 00:00:00 ~ 2025-05-01 00:00:00
국적 수: 61, 목적 수: 4

[2/4] 사용자 설정 입력...
외국인 입국자 예측 분석 - 브리지 모델링 시스템
현재 데이터: 2025년 5월까지

정상화 방법 선택:
  1: 코로나 기간 제외 (2020-2023 완전 삭제)
  2: 코로나 포함 전체 (원본 데이터 그대로)
  3: 브리지 모델링 (정상상태 연결)
  엔터: 전체 방법 (1, 2, 3 모두 실행)

선택하세요 (1, 2, 3 또는 엔터=전체): 
선택됨: 전체 정상화 방법 (1, 2, 3)

[3/4] 브리지 모델링 실험 실행...
브리지 모델링 실험 시작
총 9개 작업 예정 (정상화 3개 × 모델 3개)

정상화 방법 1: 코로나 기간 제외
------------------------------------------------------------
방법 1 적용: 59780 → 48068개 (코로나 기간 11712개 제외)
필터링 결과:
  전체 데이터: 48,068개
  훈련 데이터: 48,068개
  예측 대상: 7,564개 (2025년 6월 ~ 2027년 12월)
  국적×목적 조합: 244개

[Prophet + 정상화1] 실행 중... (1/9, 11.1%)
Prophet 모델 훈련 중 (정상화 방법 1)...


Prophet 정상화1:  11%|███████                                                         | 27/244 [00:01<00:13, 15.87it/s]

  완료: 예측 수 7,564개, 시간 14.31초

[SARIMA + 정상화1] 실행 중... (2/9, 22.2%)
SARIMA 모델 훈련 중 (정상화 방법 1)...
  계산 시간 단축을 위해 8개 그룹만 샘플링
  에러: SARIMA + 정상화1 - 처리 실패

[ETS + 정상화1] 실행 중... (3/9, 33.3%)
ETS 모델 훈련 중 (정상화 방법 1)...
  계산 시간 단축을 위해 8개 그룹만 샘플링
  에러: ETS + 정상화1 - 처리 실패

정상화 방법 2: 코로나 포함 전체
------------------------------------------------------------
방법 2 적용: 전체 59780개 데이터 사용 (원본 그대로)
필터링 결과:
  전체 데이터: 59,780개
  훈련 데이터: 59,780개
  예측 대상: 7,564개 (2025년 6월 ~ 2027년 12월)
  국적×목적 조합: 244개

[Prophet + 정상화2] 실행 중... (4/9, 44.4%)
Prophet 모델 훈련 중 (정상화 방법 2)...
  에러: Prophet + 정상화2 - 처리 실패

[SARIMA + 정상화2] 실행 중... (5/9, 55.6%)
SARIMA 모델 훈련 중 (정상화 방법 2)...
  계산 시간 단축을 위해 8개 그룹만 샘플링
  에러: SARIMA + 정상화2 - 처리 실패

[ETS + 정상화2] 실행 중... (6/9, 66.7%)
ETS 모델 훈련 중 (정상화 방법 2)...
  계산 시간 단축을 위해 8개 그룹만 샘플링
  에러: ETS + 정상화2 - 처리 실패

정상화 방법 3: 브리지 모델링
------------------------------------------------------------
방법 3 적용: 정교한 브리지 모델링 시작...
244개 그룹을 4개 워커로 병렬 처리 중...
오류 발생: I/O operation on closed file.

작업 완료: 2025-07-22 